# 📘 Módulo 05: Modificación de Datos (DML), Transacciones ACID & Niveles de Aislamiento
### 💾 Dominio Completo de INSERT, UPDATE, DELETE, MERGE (Upsert), Propiedades ACID, Control Transaccional (BEGIN, COMMIT, ROLLBACK, SAVEPOINT) y los 4 Niveles de Aislamiento ANSI SQL
**Asignatura:** Bases de Datos Avanzadas  
**Entorno:** Google Colab / DuckDB (100% In-Memory / Zero 404 Error)  

---
## 🎯 Objetivos de Aprendizaje
1. **Dominar las operaciones DML avanzadas:** `INSERT`, `UPDATE`, `DELETE`, `TRUNCATE` y sintaxis de actualización idempotente `MERGE` / `INSERT ... ON CONFLICT` (Upsert).
2. **Comprender la fundamentación teórica de las Propiedades ACID:** Atomicidad, Consistencia, Aislamiento y Durabilidad en sistemas de gestión de bases de datos relacionales.
3. **Ejecutar control transaccional explícito:** Controlar bloques de ejecución mediante `BEGIN TRANSACTION`, `COMMIT`, `ROLLBACK` y `SAVEPOINT`.
4. **Analizar las anomalías de concurrencia y Niveles de Aislamiento ANSI SQL:** Evaluar *Dirty Reads*, *Non-Repeatable Reads* y *Phantom Reads* en `READ UNCOMMITTED`, `READ COMMITTED`, `REPEATABLE READ` y `SERIALIZABLE`.
5. **Resolver casos reales de negocio:** Diseñar transferencias bancarias seguras y actualización masiva de inventarios financieros sin inconsistencias.

In [ ]:
# ==============================================================================
# 🚀 SETUP INICIAL AUTO-CONTENIDO: DUCKDB & GOOGLE COLAB (100% IN-MEMORY)
# ==============================================================================
!pip install duckdb duckdb-engine jupysql -q

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# 1. Conectar a motor DuckDB en memoria
conn = duckdb.connect(database=':memory:')

# 2. Poblar esquema transaccional de prueba (Cuentas Bancarias & Inventario)
conn.execute("""
-- Tabla de Cuentas Bancarias para Pruebas ACID
CREATE TABLE IF NOT EXISTS cuentas_bancarias (
    cuenta_id INT PRIMARY KEY,
    titular VARCHAR NOT NULL,
    saldo_actual DECIMAL(15,2) NOT NULL CHECK (saldo_actual >= 0),
    tipo_cuenta VARCHAR DEFAULT 'Ahorros'
);

INSERT INTO cuentas_bancarias VALUES
(101, 'Carlos Mendoza', 5000000.00, 'Nómina'),
(102, 'Ana María Gómez', 8500000.00, 'Ahorros'),
(103, 'Sofía Ramírez', 1200000.00, 'Corriente'),
(104, 'Empresa CUN Tech S.A.S.', 45000000.00, 'Empresarial');

-- Tabla de Log Transaccional
CREATE TABLE IF NOT EXISTS transacciones_log (
    transaccion_id INT PRIMARY KEY,
    cuenta_origen INT,
    cuenta_destino INT,
    monto DECIMAL(15,2),
    fecha_hora TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    estado VARCHAR
);

-- Tabla de Productos / Inventario para Ejercicios Upsert
CREATE TABLE IF NOT EXISTS productos_inventario (
    producto_id INT PRIMARY KEY,
    nombre_producto VARCHAR NOT NULL,
    precio_unitario DECIMAL(15,2) CHECK (precio_unitario > 0),
    stock_disponible INT CHECK (stock_disponible >= 0)
);

INSERT INTO productos_inventario VALUES
(301, 'Servidor Cloud Enterprise', 12500000.00, 10),
(302, 'Laptop Pro Developer 16', 8200000.00, 25),
(303, 'Licencia Base de Datos Pro', 4500000.00, 50);
""")

%sql conn
print('✅ Entorno DuckDB inicializado correctamente para el Módulo 05 (DML & Transacciones).')

## 1. Operaciones DML Avanzadas (Data Manipulation Language)

El sublenguaje **DML** nos permite insertar, modificar y eliminar datos en las relaciones existentes. A diferencia del DDL (`CREATE`, `ALTER`), las operaciones DML alteran el *estado* o contenido de la base de datos sin cambiar su estructura.

```mermaid
flowchart TD
    DML["Comandos DML"] --> INS["INSERT: Agregar Tuplas"]
    DML --> UPD["UPDATE: Modificar Valores"]
    DML --> DEL["DELETE: Eliminar Filas Filtradas"]
    DML --> MER["MERGE / UPSERT: Modificar o Insertar"]
```

### 1.1 Inserción de Datos (`INSERT INTO`)
Existen tres formas estándar de insertar datos:
1. Inserción fila por fila.
2. Inserción masiva multi-fila (*Bulk Insert*).
3. Inserción basada en consulta (`INSERT INTO ... SELECT`).

In [ ]:
%%sql
-- Inserción Multi-Fila en Cuentas Bancarias
INSERT INTO cuentas_bancarias (cuenta_id, titular, saldo_actual, tipo_cuenta) VALUES
(105, 'Diego Alejandro Silva', 3400000.00, 'Ahorros'),
(106, 'Valentina Morales', 6700000.00, 'Corriente');

SELECT * FROM cuentas_bancarias;

### 1.2 Actualización de Datos (`UPDATE`)
Modifica valores de atributos en tuplas existentes. **Regla de oro:** Siempre incluya la cláusula `WHERE` a menos que desee actualizar toda la tabla.

In [ ]:
%%sql
-- Aplicar un abono del 5% de intereses a todas las cuentas de Ahorros con saldo mayor a 3,000,000
UPDATE cuentas_bancarias
SET    saldo_actual = saldo_actual * 1.05
WHERE  tipo_cuenta = 'Ahorros' AND saldo_actual > 3000000;

SELECT cuenta_id, titular, saldo_actual, tipo_cuenta FROM cuentas_bancarias WHERE tipo_cuenta = 'Ahorros';

### 1.3 Borrado de Datos (`DELETE` vs `TRUNCATE`)
- `DELETE`: Operación DML que borra filas específicas según el `WHERE`. Puede ser revertida (`ROLLBACK`) dentro de una transacción.
- `TRUNCATE`: Operación DDL (en la mayoría de motores) que vacía toda la tabla instantáneamente liberando páginas de memoria.

In [ ]:
%%sql
-- Eliminar cuentas de prueba con saldo 0 o inactivas
DELETE FROM cuentas_bancarias
WHERE  saldo_actual = 0;

SELECT COUNT(*) AS total_cuentas_activas FROM cuentas_bancarias;

### 1.4 Operación Upsert / Idempotente (`INSERT ... ON CONFLICT` / `MERGE`)
En arquitecturas de integración de datos (ETL/ELT), es común requerir: *"Si el registro ya existe (por PK), actualícelo; si no existe, insértelo"*.

In [ ]:
%%sql
-- Ejemplo Upsert en DuckDB / PostgreSQL / SQLite:
-- Si el producto 301 existe, actualiza stock; si no existe, lo inserta.
INSERT INTO productos_inventario (producto_id, nombre_producto, precio_unitario, stock_disponible)
VALUES (301, 'Servidor Cloud Enterprise v2', 13000000.00, 18)
ON CONFLICT (producto_id) DO UPDATE SET
    precio_unitario = EXCLUDED.precio_unitario,
    stock_disponible = EXCLUDED.stock_disponible;

SELECT * FROM productos_inventario WHERE producto_id = 301;

## 2. Fundamentación Teórica de Transacciones ACID

Una **transacción** es una unidad lógica de trabajo (*Logical Unit of Work - LUW*) que consta de una o más instrucciones SQL ejecutadas como una sola operación indivisible.

### 🛡️ Las 4 Propiedades ACID

| Propiedad | Definición Teórica | Mecanismo del Motor SQL | Ejemplo Práctico en Banca |
| :--- | :--- | :--- | :--- |
| **A - Atomicity** *(Atomicidad)* | "Todo o Nada". La transacción se ejecuta completamente o no se ejecuta en absoluto. | Undo Log / Rollback Segment | En una transferencia, si falla la acreditación en la cuenta B, se revierte la deducción en la cuenta A. |
| **C - Consistency** *(Consistencia)* | La transacción transforma la base de datos de un estado válido a otro estado válido, respetando todas las restricciones (`PK`, `FK`, `CHECK`). | Verificación de Constraints en tiempo de ejecución | Ningún saldo de cuenta bancaria puede quedar en valor negativo si existe un `CHECK (saldo >= 0)`. |
| **I - Isolation** *(Aislamiento)* | Las transacciones concurrentes se ejecutan sin interferir entre sí como si fueran secuenciales. | Bloqueos (*Locks*) y Control de Concurrencia Multiversión (*MVCC*) | Dos compras simultáneas sobre el mismo producto no pueden vender el mismo stock dos veces. |
| **D - Durability** *(Durabilidad)* | Una vez confirmado (`COMMIT`), los cambios son permanentes incluso si ocurre una falla eléctrica o caída del sistema. | Write-Ahead Logging (*WAL*) y Redo Logs | El saldo acreditado persiste en disco tras confirmación del cajero automático. |

## 3. Control Transaccional Práctico (`BEGIN`, `COMMIT`, `ROLLBACK`, `SAVEPOINT`)

```mermaid
stateDiagram-v2
    [*] --> Activa : BEGIN TRANSACTION
    Activa --> ParcialmenteConfirmada : Última instrucción ejecutada
    ParcialmenteConfirmada --> Confirmada : COMMIT (Cambios en Disco)
    Activa --> Fallida : Error / Violación Constraint
    ParcialmenteConfirmada --> Fallida : Error de Redo Log
    Fallida --> Abortada : ROLLBACK (Deshacer cambios)
    Confirmada --> [*]
    Abortada --> [*]
```

### 3.1 Transacción Exitosa (`COMMIT`)
Demostración de transferencia bancaria de $1,000,000 desde la cuenta 101 (Carlos) hacia la cuenta 103 (Sofía).

In [ ]:
%%sql
-- Inicio del Bloque Transaccional en DuckDB
BEGIN TRANSACTION;

-- Paso 1: Restar saldo de cuenta origen (Carlos Mendoza 101)
UPDATE cuentas_bancarias
SET    saldo_actual = saldo_actual - 1000000.00
WHERE  cuenta_id = 101;

-- Paso 2: Sumar saldo a cuenta destino (Sofía Ramírez 103)
UPDATE cuentas_bancarias
SET    saldo_actual = saldo_actual + 1000000.00
WHERE  cuenta_id = 103;

-- Paso 3: Registrar en el log transaccional
INSERT INTO transacciones_log (transaccion_id, cuenta_origen, cuenta_destino, monto, estado)
VALUES (1, 101, 103, 1000000.00, 'EXITOSA');

-- Confirmación permanente de la transacción
COMMIT;

SELECT cuenta_id, titular, saldo_actual FROM cuentas_bancarias WHERE cuenta_id IN (101, 103);

### 3.2 Transacción Fallida y Reversión Automática (`ROLLBACK`)
Intentemos retirar $10,000,000 de la cuenta 103 (que solo tiene $2,200,000). Al violar el `CHECK (saldo_actual >= 0)`, la transacción aborta y se debe hacer `ROLLBACK`.

In [ ]:
# Demostración de captura de excepción y Rollback en Python/DuckDB
try:
    conn.execute("BEGIN TRANSACTION;")
    conn.execute("UPDATE cuentas_bancarias SET saldo_actual = saldo_actual - 10000000.00 WHERE cuenta_id = 103;")
    conn.execute("COMMIT;")
except Exception as e:
    conn.execute("ROLLBACK;")
    print(f"❌ ERROR DETECTADO: {e}")
    print("🔄 Transacción revertida completamente mediante ROLLBACK. La base de datos conserva su estado consistente.")

In [ ]:
%%sql
-- Verificar que el saldo de Sofía Ramírez (103) no cambió tras el fallo
SELECT cuenta_id, titular, saldo_actual FROM cuentas_bancarias WHERE cuenta_id = 103;

## 4. Anomalías de Concurrencia & Niveles de Aislamiento ANSI SQL

Cuando múltiples transacciones se ejecutan en paralelo, pueden ocurrir tres anomalías de lectura si no hay aislamiento adecuado:

### ⚠️ Anomalías de Concurrencia
1. **Lectura Sucia (*Dirty Read*):** La Transacción A lee un dato modificado por la Transacción B que aún *no ha sido confirmado* (`COMMIT`). Si B hace `ROLLBACK`, A leyó un dato fantasma que nunca existió físicamente.
2. **Lectura No Repetible (*Non-Repeatable Read*):** La Transacción A lee una fila. La Transacción B modifica o borra esa fila y confirma (`COMMIT`). Si A vuelve a leer la misma fila, obtiene valores diferentes.
3. **Lectura Fantasma (*Phantom Read*):** La Transacción A ejecuta una consulta con filtro de rango (ej. `WHERE saldo > 5M`). La Transacción B *inserta* una nueva fila que cumple dicho filtro y confirma. Cuando A vuelve a consultar, aparecen filas "fantasma" adicionales.

--- 

### 📊 Matriz ANSI SQL de Niveles de Aislamiento

| Nivel de Aislamiento | Permite Dirty Read? | Permite Non-Repeatable Read? | Permite Phantom Read? | Mecanismo Típico |
| :--- | :---: | :---: | :---: | :--- |
| **READ UNCOMMITTED** | 🔴 Sí | 🔴 Sí | 🔴 Sí | Sin bloqueos de lectura (Bajo rendimiento estricto) |
| **READ COMMITTED** | 🟢 No | 🔴 Sí | 🔴 Sí | Bloqueo corto de lectura (Default en PostgreSQL/Oracle) |
| **REPEATABLE READ** | 🟢 No | 🟢 No | 🔴 Sí | Shared Locks retenidos hasta el COMMIT |
| **SERIALIZABLE** | 🟢 No | 🟢 No | 🟢 No | Bloqueo de rangos / Snapshot Isolation (Strict ACID) |

## 5. Retos Prácticos & Casos de Estudio de Negocio

A continuación resolveremos 3 escenarios reales de arquitectura transaccional.

### 🧪 Reto 1: Procesamiento Transaccional de Pedido de Compra con Descuento de Stock
**Enunciado:**  
Un cliente realiza la compra de 2 unidades del producto `302` ('Laptop Pro Developer 16') por valor total de $16,400,000.  
Escriba un bloque transaccional que:  
1. Verifique que exista stock suficiente en `productos_inventario` (`stock_disponible >= 2`).  
2. Descuente 2 unidades del stock del producto `302`.  
3. Confirme la operación con `COMMIT`. Si no hay stock suficiente, revierta con `ROLLBACK`.

In [ ]:
# Solución Implementada en Python/DuckDB para Reto 1
def procesar_compra(producto_id, cantidad_comprada):
    try:
        conn.execute("BEGIN TRANSACTION;")
        
        # Obtener stock actual
        res = conn.execute("SELECT stock_disponible FROM productos_inventario WHERE producto_id = ?", [producto_id]).fetchone()
        if not res:
            raise Exception("Producto no encontrado.")
            
        stock_actual = res[0]
        if stock_actual < cantidad_comprada:
            raise Exception(f"Stock insuficiente. Disponible: {stock_actual}, Solicitado: {cantidad_comprada}")
            
        # Descontar stock
        conn.execute("UPDATE productos_inventario SET stock_disponible = stock_disponible - ? WHERE producto_id = ?", [cantidad_comprada, producto_id])
        conn.execute("COMMIT;")
        print(f"✅ COMPRA EXITOSA: Se descontaron {cantidad_comprada} unidades del producto {producto_id}.")
    except Exception as err:
        conn.execute("ROLLBACK;")
        print(f"❌ ERROR EN COMPRA: {err}. Transacción revertida.")

# Prueba 1: Compra Válida (2 Laptops)
procesar_compra(302, 2)

# Prueba 2: Compra Inválida (Excede stock)
procesar_compra(302, 500)

In [ ]:
%%sql
-- Estado final del inventario tras la prueba transaccional
SELECT producto_id, nombre_producto, stock_disponible FROM productos_inventario WHERE producto_id = 302;

## 📌 Resumen de Competencias Adquiridas en el Módulo 05

- [x] **DML Avanzado:** Dominio de `INSERT`, `UPDATE`, `DELETE` y `MERGE / UPSERT` (`ON CONFLICT`).
- [x] **Propiedades ACID:** Fundamentos de Atomicidad, Consistencia, Aislamiento y Durabilidad.
- [x] **Bloques Transaccionales:** Control explícito con `BEGIN TRANSACTION`, `COMMIT` y `ROLLBACK` frente a excepciones.
- [x] **Matriz de Concurrencia:** Distinción rigurosa de *Dirty Reads*, *Non-Repeatable Reads* y *Phantom Reads* en los 4 niveles de aislamiento ANSI SQL.
- [x] **Garantía In-Memory:** Ejecución 100% interactiva en Google Colab con DuckDB sin dependencias externas (Zero 404 Error).